In [0]:



df_raw= spark.read.format("delta").load("/Volumes/workspace/default/api/landing/")
df_raw.show()




from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType()),
    StructField("dt", LongType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("humidity", IntegerType())
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("description", StringType())
    ])))
])

from pyspark.sql.functions import from_json, col

df_parsed = df_raw.withColumn(
    "json_data",
    from_json(col("raw_data"), schema)
)

df_parsed.printSchema()

df_clean = df_parsed.select(
    col("json_data.name").alias("city"),
    col("json_data.main.temp").alias("temperature"),
    col("json_data.main.humidity").alias("humidity"),
    col("json_data.weather")[0]["description"].alias("weather")
)

display(df_clean)